#  Script 3 — Labeling + Naive Bayes + Evaluasi
## Analisis Sentimen Konten Dakwah Penceramah Kota Bandung
**Author:** Fahmi Faturahman — STMIK Bandung 2026

---
###  Alur Script:
```
CELL 1 → Install library
CELL 2 → Mount Google Drive & Load data bersih
CELL 3 → Labeling Semi-Otomatis (Lexicon-Based) — BINARY: Positif/Negatif saja
CELL 4 → Cek & validasi distribusi label (2 kelas)
CELL 5 → Preprocessing lanjutan (Tokenizing + Stemming)
CELL 6 → TF-IDF Feature Extraction
CELL 7 → Split data Train / Test
CELL 8 → Training Multinomial Naive Bayes
CELL 9 → Evaluasi: Confusion Matrix + Classification Report
CELL 10 → Visualisasi hasil evaluasi
CELL 11 → Simpan model & hasil
```
### Input & Output:
- **Input:**  `clean_data_komentar.csv` (output dari Script 2)
- **Output:** `model_naive_bayes.pkl` + `hasil_prediksi.csv` + grafik evaluasi

>  **Jalankan Script 1 dan Script 2 terlebih dahulu!**

---
## Cell 1 — Install Library
> Jalankan sekali di setiap sesi Colab baru.

In [1]:
!pip install -q PySastrawi
!pip install -q scikit-learn
!pip install -q pandas numpy matplotlib seaborn
!pip install -q openpyxl
# 🆕 BARU — Untuk download InSet Lexicon dari GitHub
!pip install -q requests

print('✅ Semua library berhasil diinstall!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 3.7 MB/s eta 0:00:00
✅ Semua library berhasil diinstall!


---
## Cell 2 — Mount Google Drive & Load Data
> Load data bersih hasil dari Script 2.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE_DIR       = '/content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2'
INPUT_CSV      = os.path.join(BASE_DIR, 'clean_data_komentar.csv')
OUTPUT_DIR     = os.path.join(BASE_DIR, 'hasil_model_binary_lexicon_tanpa_netral')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 🆕 BARU — Path checkpoint stemming (resume otomatis jika Colab disconnect)
PATH_STEMMED   = os.path.join(BASE_DIR, 'clean_data_stemmed.csv')

# 🆕 BARU — Path cache InSet Lexicon (download sekali, disimpan di Drive)
PATH_INSET_POS = os.path.join(BASE_DIR, 'inset_lexicon_positive.tsv')
PATH_INSET_NEG = os.path.join(BASE_DIR, 'inset_lexicon_negative.tsv')

# Load data
df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')

print('='*55)
print(' DATA YANG DIMUAT')
print('='*55)
print(f'Total komentar     : {len(df):,}')
print(f'Kolom tersedia     : {list(df.columns)}')
print()
print('5 baris pertama:')
display(df[['comment_text', 'comment_clean']].head())

Mounted at /content/drive
 DATA YANG DIMUAT
Total komentar     : 18,910
Kolom tersedia     : ['comment_id', 'comment_text', 'author_name', 'like_count', 'published_at', 'video_id', 'channel_name', 'topik_utama', 'panjang_raw', 'jumlah_kata_raw', 'comment_clean', 'comment_no_stopword', 'panjang_clean', 'jumlah_kata_clean', 'tokens']

5 baris pertama:


,comment_text,comment_clean
0,"Andai saja hadir juga : Aa Gym, Buya Yahya, Ha...",andai saja hadir juga aa gym buya yahya habib ...
1,Saya terlahir NU dan tumbuh dg segala aturan d...,saya terlahir nu dan tumbuh dengan segala atur...
2,YANG NONTON INI SAMBIL NANGISS😭☝🏻☝🏻☝🏻 Allahumm...,yang menonton ini sambil menangis allahumma sh...
3,saya bersaksi guru guru kami adalah orang oran...,saya bersaksi guru guru kami adalah orang oran...
4,Teringat caption pada foto para guru ini bbrp ...,teringat caption pada foto para guru ini beber...


---
## Cell 3 —  Labeling Semi-Otomatis (InSet Lexicon) — **BINARY (Positif/Negatif)**

> ### Cara kerja labeling:
> Script ini menggunakan **InSet Lexicon** (Fachrina et al., 2016) sebagai kamus sentimen utama,
> diperkuat dengan ekstensi kosakata domain dakwah Islam.
>
> **Sumber kamus:**
> - **InSet Lexicon** — ~3.600 kata positif + ~6.600 kata negatif (total ~10.218 kata)
> - **Ekstensi Dakwah** — kosakata khas konten ceramah yang tidak ada di InSet
>   (masyaallah, subhanallah, barakallah, sesat, bidah, dll)
>
> **Fitur tambahan:**
> -  Deteksi negasi — `"tidak bagus"` → Negatif (bukan Positif)
> -  Matching n-gram longest-match (frasa sampai `MAX_NGRAM` kata)
> -  Lookup O(1) menggunakan Python `set`
> -  Overlap kata dihapus otomatis (kata ambigu tidak dipakai)
>
> ### 🔴 Perubahan: kelas Netral DIHAPUS
> Penelitian ini hanya memakai **dua kelas: Positif dan Negatif**.
> Komentar dengan `skor_pos == skor_neg` (seri — termasuk komentar yang tidak
> mengandung satu pun kata sentimen) **tidak lagi dilabeli Netral**, melainkan
> dianggap **tidak terklasifikasi** dan langsung dibuang dari dataset di cell ini.
>
> Komentar yang dibuang tetap diekspor ke `komentar_tidak_berlabel.csv`
> supaya jumlahnya bisa dilaporkan di naskah (transparansi data).
>
> **Referensi:**
> Fachrina, W. O., Widyantoro, D. H., & Suhartono, D. (2016).
> *Indonesian Sentiment Lexicon (InSet).*


In [3]:
import requests, os, re
import pandas as pd


PATH_KAMUS_POS = os.path.join(BASE_DIR, 'kamus_gabungan_positive.tsv')
PATH_KAMUS_NEG = os.path.join(BASE_DIR, 'kamus_gabungan_negative.tsv')

print('=' * 60)
print('📚 MEMUAT Kamus Gabungan (InSet + Ekstensi Dakwah)')
print('=' * 60)

df_pos = pd.read_csv(PATH_KAMUS_POS, sep='\t', encoding='utf-8')
df_neg = pd.read_csv(PATH_KAMUS_NEG, sep='\t', encoding='utf-8')

KATA_POSITIF_FINAL = set(df_pos['kata'].str.lower().str.strip())
KATA_NEGATIF_FINAL = set(df_neg['kata'].str.lower().str.strip())

# Hapus overlap — jika kata ada di kedua kamus, hapus dari keduanya
# (kata ambigu lebih baik tidak dipakai daripada salah label)
overlap = KATA_POSITIF_FINAL & KATA_NEGATIF_FINAL
if overlap:
    KATA_POSITIF_FINAL -= overlap
    KATA_NEGATIF_FINAL -= overlap

print()
print('── Rincian per Sumber ───────────────────────────────────')
for label_sisi, df_sisi in [('Positif', df_pos), ('Negatif', df_neg)]:
    hitung = df_sisi['sumber'].value_counts()
    for sumber, n in hitung.items():
        print(f'  {label_sisi:<8} - {sumber:<14}: {n:,} kata')
print()
print('── Kamus Final (setelah overlap dihapus) ────────────────')
print(f'  KATA_POSITIF_FINAL  : {len(KATA_POSITIF_FINAL):,} kata')
print(f'  KATA_NEGATIF_FINAL  : {len(KATA_NEGATIF_FINAL):,} kata')
print(f'  TOTAL               : {len(KATA_POSITIF_FINAL) + len(KATA_NEGATIF_FINAL):,} kata')
if overlap:
    print(f'  Kata overlap dihapus: {len(overlap)} kata')
print()


# ── Fungsi Labeling ───────────────────────────────────────────
# Kata negasi yang membalik polaritas kata setelahnya
KATA_NEGASI = {
    'tidak', 'tak', 'bukan', 'jangan', 'belum', 'tanpa',
    'ga', 'gak', 'enggak', 'nggak', 'ndak', 'kagak',
}

MAX_NGRAM = max(
    max((len(k.split()) for k in KATA_POSITIF_FINAL), default=1),
    max((len(k.split()) for k in KATA_NEGATIF_FINAL), default=1),
)


def hitung_skor_sentimen(text: str):
    """
    Hitung skor positif & negatif sebuah komentar berbasis kamus gabungan.

    Fitur:
    - Cek N-GRAM longest-match (dari MAX_NGRAM kata turun ke 1 kata),
      karena kamus punya entri sampai 4 kata (mis. "islam rahmatan lil alamin")
    - Deteksi negasi sederhana (tidak bagus → dihitung negatif)
    - Bobot: setiap match dihitung +1 ke sisi yang cocok

    Returns:
        (skor_pos, skor_neg)
    """
    if not isinstance(text, str) or not text.strip():
        return 0, 0

    tokens = text.lower().split()
    n = len(tokens)
    skor_pos = 0
    skor_neg = 0

    i = 0
    while i < n:
        cocok = False
        # Coba frasa terpanjang dulu, turun ke 1 kata (longest-match)
        for panjang in range(min(MAX_NGRAM, n - i), 0, -1):
            frasa = ' '.join(tokens[i:i + panjang])
            if frasa not in KATA_POSITIF_FINAL and frasa not in KATA_NEGATIF_FINAL:
                continue
            ada_negasi = (i > 0) and (tokens[i - 1] in KATA_NEGASI)
            if frasa in KATA_POSITIF_FINAL:
                skor_neg += 1 if ada_negasi else 0
                skor_pos += 0 if ada_negasi else 1
            else:
                skor_pos += 1 if ada_negasi else 0
                skor_neg += 0 if ada_negasi else 1
            i += panjang
            cocok = True
            break
        if not cocok:
            i += 1

    return skor_pos, skor_neg


def label_sentimen(text: str):
    """
    Labeling BINARY — hanya 'Positif' atau 'Negatif'.

    🔴 Kelas Netral sudah DIHAPUS dari penelitian ini.
       Kalau skor seri (skor_pos == skor_neg) — termasuk komentar yang tidak
       mengandung satu pun kata sentimen — fungsi mengembalikan None, artinya
       komentar itu TIDAK TERKLASIFIKASI dan akan dibuang dari dataset,
       bukan dilempar ke kelas Netral.

    Returns:
        'Positif' | 'Negatif' | None
    """
    skor_pos, skor_neg = hitung_skor_sentimen(text)

    if skor_pos > skor_neg:
        return 'Positif'
    elif skor_neg > skor_pos:
        return 'Negatif'
    else:
        return None


# ── Jalankan Labeling ─────────────────────────────────────────
print('=' * 60)
print('🏷️  Menjalankan labeling BINARY (Positif/Negatif)...')
print('=' * 60)

df['sentiment_label'] = df['comment_clean'].apply(label_sentimen)

# ── Buang komentar yang tidak terklasifikasi (skor seri) ──────
n_sebelum   = len(df)
mask_kosong = df['sentiment_label'].isna()
n_dibuang   = int(mask_kosong.sum())

# Ekspor komentar terbuang dulu (buat dilaporkan di naskah / lampiran)
if n_dibuang > 0:
    kolom_ekspor = [k for k in ['comment_id', 'comment_text', 'comment_clean',
                                'channel_name', 'video_id'] if k in df.columns]
    path_tak_berlabel = os.path.join(OUTPUT_DIR, 'komentar_tidak_berlabel.csv')
    df.loc[mask_kosong, kolom_ekspor].to_csv(
        path_tak_berlabel, index=False, encoding='utf-8-sig'
    )
    print(f'📄 Komentar tidak terklasifikasi diekspor ke: {path_tak_berlabel}')

df = df[~mask_kosong].reset_index(drop=True)

print()
print('✅ Labeling selesai!')
print()
print(f'  Komentar sebelum labeling      : {n_sebelum:,}')
print(f'  Dibuang (skor seri/tanpa kata) : {n_dibuang:,} ({n_dibuang / n_sebelum * 100:.1f}%)')
print(f'  Komentar terpakai (2 kelas)    : {len(df):,}')
print()

print('📊 Distribusi Label:')
dist = df['sentiment_label'].value_counts()
for label, jumlah in dist.items():
    pct = jumlah / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<10} : {jumlah:>6,} ({pct:>5.1f}%) {bar}')

# Pengaman: pastikan benar-benar cuma 2 kelas, tidak ada Netral nyelip
kelas_ada = set(df['sentiment_label'].unique())
assert kelas_ada <= {'Positif', 'Negatif'}, f'Masih ada kelas lain: {kelas_ada}'
print()
print(f'🔒 Verifikasi kelas: {sorted(kelas_ada)} — tidak ada Netral. Aman.')

print()
print('📋 Contoh hasil labeling:')
display(df[['comment_text', 'comment_clean', 'sentiment_label']].sample(10, random_state=42))


📚 MEMUAT Kamus Gabungan (InSet + Ekstensi Dakwah)

── Rincian per Sumber ───────────────────────────────────
  Positif  - inset         : 3,609 kata
  Positif  - dakwah_custom : 173 kata
  Negatif  - inset         : 6,609 kata
  Negatif  - dakwah_custom : 197 kata

── Kamus Final (setelah overlap dihapus) ────────────────
  KATA_POSITIF_FINAL  : 2,633 kata
  KATA_NEGATIF_FINAL  : 5,656 kata
  TOTAL               : 8,289 kata
  Kata overlap dihapus: 1147 kata

🏷️  Menjalankan labeling BINARY (Positif/Negatif)...
📄 Komentar tidak terklasifikasi diekspor ke: /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/komentar_tidak_berlabel.csv

✅ Labeling selesai!

  Komentar sebelum labeling      : 18,910
  Dibuang (skor seri/tanpa kata) : 3,259 (17.2%)
  Komentar terpakai (2 kelas)    : 15,651

📊 Distribusi Label:
  Positif    :  9,774 ( 62.4%) ███████████████████████████████
  Negatif    :  5,877 ( 37.6%) ██████████████████

🔒 Verifika

,comment_text,comment_clean,sentiment_label
2009,"Mohon maaf, izin menyampaikan pendapat.. \nDul...",mohon maaf izin menyampaikan pendapat dulu say...,Positif
169,"Alhamdulillah, semoga berkah❤❤❤",alhamdulillah semoga berkah,Positif
1402,dari ekspresi nyengirnya pas blg kuntilanak ka...,dari ekspresi nyengirnya pas bilang kuntilanak...,Negatif
2126,"Negeri ini kaya raya, kekayaan alamnya melimpa...",negeri ini kayak raya kekayaan alamnya melimpa...,Positif
10069,Tolong fong dlm tausyah nya mengambil tema nya...,tolong fong dalam tausyah nya mengambil tema n...,Negatif
11267,Ya Allah sehatkan sekeluarga Ustad Das'ad agar...,ya allah sehatkan sekeluarga ustad das ada aga...,Positif
1662,Kenapa malah mengakui penjajah !? Sedangkan du...,kenapa malah mengakui penjajah sedangkan dulu ...,Negatif
4962,"Obrolan yang sangat menyentuh hati,semoga yang...",obrolan yang sangat menyentuh hati semoga yang...,Negatif
4176,"Assalamualaikum warahmatullahi wabarakatuh,, s...",assalamualaikum warahmatullahi wabarakatuh seh...,Positif
2214,"Tinggal tunggu waktu saja, sekuat apa rakyat b...",tinggal tunggu waktu saja sekuat apa rakyat bi...,Negatif


---
## Cell 4 — Cek & Validasi Distribusi Label


In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 130})

dist = df['sentiment_label'].value_counts()
total = len(df)

print('='*55)
print('📊 DISTRIBUSI LABEL SENTIMEN')
print('='*55)
for label, n in dist.items():
    pct = n / total * 100
    print(f'  {label:<12}: {n:>6,} ({pct:.1f}%)')
print(f'  {"─"*35}')
print(f'  Total       : {total:>6,} (100.0%)')

# Cek imbalance
dominan_pct = dist.max() / total * 100
print()
if dominan_pct > 80:
    print(f'  PERINGATAN: Kelas "{dist.idxmax()}" terlalu dominan ({dominan_pct:.1f}%)')
    print('   Pertimbangkan untuk memperluas kamus kata sentimen')
    print('   atau terapkan teknik SMOTE untuk menyeimbangkan kelas')
elif dominan_pct > 60:
    print(f'  INFO: Kelas "{dist.idxmax()}" agak dominan ({dominan_pct:.1f}%) — masih wajar')
else:
    print(f' Distribusi kelas cukup seimbang!')

# Pie chart distribusi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribusi Label Sentimen', fontsize=14, fontweight='bold')

colors = {'Positif': '#55A868', 'Negatif': '#C44E52'}
clr = [colors.get(l, '#999') for l in dist.index]

axes[0].pie(dist.values, labels=dist.index, autopct='%1.1f%%',
           colors=clr, startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Proporsi (%)')

bars = axes[1].bar(dist.index, dist.values, color=clr, edgecolor='white', width=0.5)
axes[1].bar_label(bars, fmt='{:,.0f}', padding=5, fontsize=11, fontweight='bold')
axes[1].set_title('Jumlah Komentar per Label')
axes[1].set_ylabel('Jumlah')
axes[1].set_ylim(0, dist.max() * 1.2)

plt.tight_layout()
path_dist = os.path.join(OUTPUT_DIR, 'grafik_distribusi_label.png')
plt.savefig(path_dist, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n Grafik disimpan: {path_dist}')

# Validasi manual sampel acak
print()
print(' SAMPEL ACAK UNTUK VALIDASI MANUAL (30 komentar):')
print('   Cek apakah label yang diberikan sudah sesuai konteks')
display(df[['comment_text', 'sentiment_label']].sample(30, random_state=99))

📊 DISTRIBUSI LABEL SENTIMEN
  Positif     :  9,774 (62.4%)
  Negatif     :  5,877 (37.6%)
  ───────────────────────────────────
  Total       : 15,651 (100.0%)

  INFO: Kelas "Positif" agak dominan (62.4%) — masih wajar

 Grafik disimpan: /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/grafik_distribusi_label.png

 SAMPEL ACAK UNTUK VALIDASI MANUAL (30 komentar):
   Cek apakah label yang diberikan sudah sesuai konteks


,comment_text,sentiment_label
8189,pawang hujan novoliayu jashujasanur moga menge...,Positif
8064,Bahasa ucapan dg bahasa menimpa diri sendiri b...,Negatif
4832,negara kok ijinkan minuman seperti itu beredar ??,Negatif
6280,"Assalamualaikum\nSehat sehat Gus iqdam, usiany...",Positif
4745,saya punya keluhannya tanggan selalu dan kebas...,Negatif
3736,Pintar berbicara tapi tak bisa intropeksi diri 😂,Negatif
13465,❤Amin3❤syg ku semuga yg terbaik dlm imanislam ...,Positif
11965,"Alhamdulilah, seneng bs ikut kajian ilmu dg Us...",Positif
4202,sehat sehat trus ustadz,Positif
14354,Alhamdulillah....akhirnya Nikah jua...❤ selama...,Positif


---
## Cell 5 —  Preprocessing Lanjutan
> Tokenizing + Stemming menggunakan Sastrawi (algoritma Nazief & Adriani).
> Ini mengubah kata berimbuhan menjadi kata dasar.
>
> Contoh: *'menyampaikan'* → *'sampai'*, *'pembelajaran'* → *'ajar'*

In [5]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import os


if os.path.exists(PATH_STEMMED):
    # ── PATH A: File sudah ada → load langsung, skip stemming ──────────────
    print('=' * 60)
    print('⚡ RESUME TERDETEKSI — Memuat hasil stemming dari Drive...')
    print('=' * 60)

    df_stemmed = pd.read_csv(PATH_STEMMED, encoding='utf-8-sig')

    # Validasi: pastikan kolom penting ada di file yang di-load
    kolom_wajib = ['comment_stemmed', 'sentiment_label']
    kolom_hilang = [k for k in kolom_wajib if k not in df_stemmed.columns]

    if kolom_hilang:
        print(f'⚠️  File resume tidak valid, kolom hilang: {kolom_hilang}')
        print('   Hapus file clean_data_stemmed.csv lalu jalankan ulang cell ini.')
    else:
        # Sinkronisasi: ambil hanya baris yang ada di df (hasil labeling Cell 3)
        # Ini penting supaya filter dari cell sebelumnya tetap berlaku
        if 'comment_id' in df_stemmed.columns and 'comment_id' in df.columns:
            df_stemmed_only = df_stemmed[['comment_id', 'comment_stemmed']].drop_duplicates('comment_id')
            df = df.merge(df_stemmed_only, on='comment_id', how='inner')
        else:
            df = df_stemmed.copy()

        # 🔒 PENGAMAN: file clean_data_stemmed.csv dari run LAMA masih menyimpan
        # kolom sentiment_label versi 3 kelas. Kalau sampai ikut kebawa, buang.
        if 'sentiment_label' in df.columns:
            n_asing = int((~df['sentiment_label'].isin(['Positif', 'Negatif'])).sum())
            if n_asing:
                print(f'⚠️  {n_asing:,} baris berlabel selain Positif/Negatif (sisa cache lama) — dibuang.')
                df = df[df['sentiment_label'].isin(['Positif', 'Negatif'])].reset_index(drop=True)

        print(f'✅ Berhasil dimuat: {len(df):,} baris')
        print(f'   Kolom tersedia : {list(df.columns)}')
        print()
        print('   ℹ️  Stemming di-skip karena file sudah ada.')
        print('   ℹ️  Hapus clean_data_stemmed.csv jika ingin stemming ulang.')
        print()
        print('📋 Contoh data yang dimuat:')
        display(df[['comment_clean', 'comment_stemmed', 'sentiment_label']].head(5))

else:
    # ── PATH B: File belum ada → jalankan stemming dari awal ───────────────
    print('=' * 60)
    print('🔄 Menjalankan stemming...')
    print('   (proses ini memakan waktu ±5-30 menit tergantung jumlah data)')
    print('   Sabar ya bro, Sastrawi memproses tiap kata satu per satu!')
    print('=' * 60)

    # Inisialisasi Sastrawi
    stemmer_factory  = StemmerFactory()
    stemmer          = stemmer_factory.create_stemmer()
    sw_factory       = StopWordRemoverFactory()
    stopword_remover = sw_factory.create_stop_word_remover()

    def stemming(text: str) -> str:
        """Stemming kata berimbuhan → kata dasar (Nazief & Adriani via Sastrawi)"""
        if not isinstance(text, str) or not text.strip():
            return ''
        return stemmer.stem(text)

    def full_preprocessing(text: str) -> str:
        """Stopword removal + Stemming — untuk kolom yang akan masuk TF-IDF"""
        if not isinstance(text, str) or not text.strip():
            return ''
        text = stopword_remover.remove(text)
        text = stemmer.stem(text)
        return text

    # Tampilkan progress tiap 10%
    total     = len(df)
    milestone = max(total // 10, 1)

    hasil_stemmed = []
    kolom_input   = 'comment_no_stopword' if 'comment_no_stopword' in df.columns else 'comment_clean'
    print(f'   Menggunakan kolom input : {kolom_input}')
    print(f'   Total data              : {total:,} baris')
    print()

    for i, text in enumerate(df[kolom_input], 1):
        hasil_stemmed.append(stemming(text))
        if i % milestone == 0 or i == total:
            pct = i / total * 100
            bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
            print(f'   [{bar}] {pct:>5.1f}%  ({i:,}/{total:,})', end='\r')

    print()  # newline setelah progress bar
    df['comment_stemmed'] = hasil_stemmed

    # Hapus baris yang jadi kosong setelah stemming
    n_sebelum = len(df)
    df = df[df['comment_stemmed'].str.strip() != ''].reset_index(drop=True)
    n_sesudah = len(df)

    # 🔧 SIMPAN CHECKPOINT — Atomic write ke Drive
    tmp_path = PATH_STEMMED + '.tmp'
    df.to_csv(tmp_path, index=False, encoding='utf-8-sig')
    os.replace(tmp_path, PATH_STEMMED)

    print(f'\n✅ Stemming selesai!')
    print(f'   Data sebelum : {n_sebelum:,}')
    print(f'   Data sesudah : {n_sesudah:,} ({n_sebelum - n_sesudah} baris kosong dihapus)')
    print(f'   💾 Checkpoint disimpan: {PATH_STEMMED}')
    print(f'      → Jika Colab disconnect, jalankan ulang cell ini')
    print(f'        untuk load dari file tanpa stemming ulang.')
    print()
    print('📋 Contoh hasil stemming:')
    display(df[['comment_clean', 'comment_stemmed', 'sentiment_label']].head(8))

⚡ RESUME TERDETEKSI — Memuat hasil stemming dari Drive...
✅ Berhasil dimuat: 15,642 baris
   Kolom tersedia : ['comment_id', 'comment_text', 'author_name', 'like_count', 'published_at', 'video_id', 'channel_name', 'topik_utama', 'panjang_raw', 'jumlah_kata_raw', 'comment_clean', 'comment_no_stopword', 'panjang_clean', 'jumlah_kata_clean', 'tokens', 'sentiment_label', 'comment_stemmed']

   ℹ️  Stemming di-skip karena file sudah ada.
   ℹ️  Hapus clean_data_stemmed.csv jika ingin stemming ulang.

📋 Contoh data yang dimuat:


,comment_clean,comment_stemmed,sentiment_label
0,andai saja hadir juga aa gym buya yahya habib ...,andai hadir aa gym buya yahya habib riziek ust...,Positif
1,saya terlahir nu dan tumbuh dengan segala atur...,lahir nu tumbuh atur tata nu berkat ustadz adi...,Positif
2,saya bersaksi guru guru kami adalah orang oran...,saksi guru guru soleh panjang umur mulia dunia...,Positif
3,teringat caption pada foto para guru ini beber...,caption foto guru tahun apa halaqah mu saudara...,Positif
4,fiqih itu luas yang sempit itu adalah hati kita,fiqih luas sempit hati,Negatif


---
## Cell 6 — TF-IDF Feature Extraction
> Mengubah teks menjadi representasi angka (vektor) yang bisa dipahami algoritma.
>
> **Analogi:** TF-IDF seperti mengubah setiap komentar menjadi 'sidik jari angka'
> yang merepresentasikan kata-kata paling penting di dalamnya.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Kolom teks yang akan dijadikan fitur
KOLOM_FITUR = 'comment_stemmed'
KOLOM_LABEL = 'sentiment_label'

# Inisialisasi TF-IDF
tfidf = TfidfVectorizer(
    max_features = 5000,    # Ambil 5000 kata dengan bobot tertinggi
    min_df       = 2,       # Kata harus muncul minimal di 2 dokumen
    max_df       = 0.95,    # Abaikan kata yang muncul di >95% dokumen
    ngram_range  = (1, 2),  # Gunakan unigram dan bigram
    sublinear_tf = True,    # Gunakan log normalization untuk TF
)

X = tfidf.fit_transform(df[KOLOM_FITUR].fillna(''))
y = df[KOLOM_LABEL]

print('='*55)
print('📐 HASIL TF-IDF VECTORIZATION')
print('='*55)
print(f'Jumlah dokumen (komentar) : {X.shape[0]:,}')
print(f'Jumlah fitur (kata/bigram): {X.shape[1]:,}')
print(f'Ukuran matriks TF-IDF     : {X.shape}')
print()
print('Top 20 kata dengan bobot TF-IDF tertinggi:')
fitur_names = tfidf.get_feature_names_out()
print(', '.join(fitur_names[:20]))

📐 HASIL TF-IDF VECTORIZATION
Jumlah dokumen (komentar) : 15,642
Jumlah fitur (kata/bigram): 5,000
Ukuran matriks TF-IDF     : (15642, 5000)

Top 20 kata dengan bobot TF-IDF tertinggi:
aa, aa gim, aa gym, aa hati, aa ilmu, aa moga, aa sehat, aaamiin, aagym, aalaamiin, aalamiin, aalamin, aali, aali sayyidina, aallah, aalloh, aam, aamiiiin, aamiiinn, aamiinn


---
## Cell 7 — Split Data Train / Test
> Membagi dataset menjadi **80% data latih** dan **20% data uji**.
>
> **Analogi:** Seperti latihan soal (80%) dan ujian sebenarnya (20%).
> Model belajar dari data latih, lalu diuji pada data yang belum pernah dilihat.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,         # 20% untuk testing
    random_state = 42,          # Seed agar hasil reproducible
    stratify     = y            # Pastikan proporsi kelas sama di train & test
)

print('='*55)
print('  HASIL SPLIT DATA')
print('='*55)
print(f'Total data          : {X.shape[0]:,}')
print(f'Data latih (80%)    : {X_train.shape[0]:,}')
print(f'Data uji   (20%)    : {X_test.shape[0]:,}')
print()
print('Distribusi kelas di data latih:')
for label, n in y_train.value_counts().items():
    print(f'  {label:<12}: {n:,} ({n/len(y_train)*100:.1f}%)')
print()
print('Distribusi kelas di data uji:')
for label, n in y_test.value_counts().items():
    print(f'  {label:<12}: {n:,} ({n/len(y_test)*100:.1f}%)')

  HASIL SPLIT DATA
Total data          : 15,642
Data latih (80%)    : 12,513
Data uji   (20%)    : 3,129

Distribusi kelas di data latih:
  Positif     : 7,816 (62.5%)
  Negatif     : 4,697 (37.5%)

Distribusi kelas di data uji:
  Positif     : 1,955 (62.5%)
  Negatif     : 1,174 (37.5%)


---
## Cell 8 — Training Multinomial Naive Bayes
> Training model klasifikasi sentimen menggunakan Multinomial Naive Bayes.
>
> Parameter **alpha** adalah Laplace Smoothing — mencegah probabilitas bernilai nol
> untuk kata yang belum pernah muncul di data latih.
> **Update:** classifier diganti dari `MultinomialNB` ke `ComplementNB` — varian Naive Bayes
> yang didesain khusus buat data imbalanced (statistiknya dihitung dari kelas komplemen,
> bukan kelas itu sendiri), diharapkan lebih adil ke kelas minoritas (Negatif).
> Alpha tetap dicari lewat `GridSearchCV` biar perbandingan sama alpha-tuning MultinomialNB adil (apple-to-apple).


In [8]:
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import time

print(' Mencari alpha terbaik utk ComplementNB lewat GridSearchCV (5-fold CV)...')
start_time = time.time()

# ── ComplementNB + Alpha Tuning ─────────────────────────────────
# ComplementNB: varian Naive Bayes yang statistiknya dihitung dari kelas KOMPLEMEN
# (bukan kelas itu sendiri) -- didesain khusus utk data imbalanced spt kasus ini
# (Positif jauh lebih banyak dari Negatif). alpha tetap di-grid biar adil dibanding MultinomialNB.
param_grid = {'alpha': [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]}
grid = GridSearchCV(
    ComplementNB(), param_grid,
    cv=5, scoring='f1_weighted', n_jobs=-1
)
grid.fit(X_train, y_train)

print(f'   Alpha terbaik      : {grid.best_params_["alpha"]}')
print(f'   F1-weighted (CV)   : {grid.best_score_:.4f}')
print()
print('   Rincian tiap alpha yang dicoba:')
for a, score in sorted(zip(grid.cv_results_['param_alpha'], grid.cv_results_['mean_test_score']), key=lambda x: x[0]):
    tanda = '  <-- terbaik' if a == grid.best_params_['alpha'] else ''
    print(f'     alpha={a:<5} -> F1-weighted CV = {score:.4f}{tanda}')
print()

# Training model final pakai alpha terbaik
nb_model = grid.best_estimator_

elapsed = time.time() - start_time
print(f'\n Training + tuning selesai dalam {elapsed:.2f} detik!')
print(f'   Algoritma  : Complement Naive Bayes')
print(f'   Alpha      : {nb_model.alpha} (hasil GridSearchCV)')
print(f'   Kelas      : {list(nb_model.classes_)}')
print(f'   Fitur      : {nb_model.n_features_in_:,}')
print()

# Prediksi pada data uji
y_pred = nb_model.predict(X_test)
y_prob = nb_model.predict_proba(X_test)

print(' Contoh 10 hasil prediksi:')
# Ambil indeks asli dari df
idx_test = y_test.index[:10]
contoh = pd.DataFrame({
    'Komentar'     : df.loc[idx_test, 'comment_text'].values,
    'Label Asli'   : y_test.values[:10],
    'Label Prediksi': y_pred[:10],
    'Benar?'       : ['✅' if a==p else '❌'
                      for a,p in zip(y_test.values[:10], y_pred[:10])]
})
display(contoh)

 Mencari alpha terbaik utk ComplementNB lewat GridSearchCV (5-fold CV)...
   Alpha terbaik      : 2.0
   F1-weighted (CV)   : 0.8216

   Rincian tiap alpha yang dicoba:
     alpha=0.01  -> F1-weighted CV = 0.8096
     alpha=0.05  -> F1-weighted CV = 0.8114
     alpha=0.1   -> F1-weighted CV = 0.8117
     alpha=0.3   -> F1-weighted CV = 0.8146
     alpha=0.5   -> F1-weighted CV = 0.8155
     alpha=0.7   -> F1-weighted CV = 0.8168
     alpha=1.0   -> F1-weighted CV = 0.8187
     alpha=1.5   -> F1-weighted CV = 0.8204
     alpha=2.0   -> F1-weighted CV = 0.8216  <-- terbaik


 Training + tuning selesai dalam 9.09 detik!
   Algoritma  : Complement Naive Bayes
   Alpha      : 2.0 (hasil GridSearchCV)
   Kelas      : [np.str_('Negatif'), np.str_('Positif')]
   Fitur      : 5,000

 Contoh 10 hasil prediksi:


,Komentar,Label Asli,Label Prediksi,Benar?
0,Assalammualaikum Madinah Menyimak Mabruuk untu...,Positif,Positif,✅
1,Pointya 1.. Sumur jgn pernah di urug total.. K...,Positif,Negatif,❌
2,MasyaAllah.. Barokallah atas ilmunya a.. Semog...,Positif,Positif,✅
3,mg dosa dosanya beliau diamuni allah swt aamiin,Positif,Positif,✅
4,Aku sedikit tahu cerita kemantaban Irene Hando...,Positif,Positif,✅
5,Ini yg saya alami beberapa bulan ini 😢 saya me...,Negatif,Positif,❌
6,semangat berbagi upgris,Positif,Positif,✅
7,Malang jawatimur drek ngaos,Negatif,Negatif,✅
8,Aku ko malah bangga bgt k teh ria yaa... dr hd...,Positif,Positif,✅
9,"Ya Allah yang Maha mendengar, lunaskanlah semu...",Positif,Positif,✅


---
## Cell 9 —  Evaluasi Model: Confusion Matrix + Classification Report
> Mengukur performa model menggunakan 4 metrik utama:
> **Accuracy, Precision, Recall, F1-Score**


In [9]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# ── Hitung semua metrik ───────────────────────────────────────
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
LABEL_KELAS = ['Positif', 'Negatif']
cm   = confusion_matrix(y_test, y_pred, labels=LABEL_KELAS)

print('='*55)
print(' HASIL EVALUASI MODEL NAIVE BAYES')
print('='*55)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}  ({prec*100:.2f}%)')
print(f'  Recall    : {rec:.4f}  ({rec*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f}  ({f1*100:.2f}%)')
print()

# Penilaian performa
if acc >= 0.85:
    print(f'🏆 Performa SANGAT BAIK! (Accuracy ≥ 85%)')
elif acc >= 0.75:
    print(f'✅ Performa BAIK (Accuracy ≥ 75%)')
elif acc >= 0.65:
    print(f'  Performa CUKUP (Accuracy ≥ 65%) — pertimbangkan optimasi')
else:
    print(f'❌ Performa KURANG (Accuracy < 65%) — perlu optimasi')

print()
print('─'*55)
print('CLASSIFICATION REPORT (per kelas):')
print('─'*55)
print(classification_report(
    y_test, y_pred,
    labels=LABEL_KELAS,
    target_names=LABEL_KELAS,
    zero_division=0
))

print('─'*55)
print('CONFUSION MATRIX:')
print('─'*55)
print('(Baris = Label Asli, Kolom = Label Prediksi)')
print(f'{"":12} Positif  Negatif')
for i, label in enumerate(LABEL_KELAS):
    print(f'{label:<12} {cm[i][0]:>7}  {cm[i][1]:>7}')

 HASIL EVALUASI MODEL NAIVE BAYES
  Accuracy  : 0.8092  (80.92%)
  Precision : 0.8180  (81.80%)
  Recall    : 0.8092  (80.92%)
  F1-Score  : 0.8113  (81.13%)

✅ Performa BAIK (Accuracy ≥ 75%)

───────────────────────────────────────────────────────
CLASSIFICATION REPORT (per kelas):
───────────────────────────────────────────────────────
              precision    recall  f1-score   support

     Positif       0.88      0.80      0.84      1955
     Negatif       0.72      0.82      0.76      1174

    accuracy                           0.81      3129
   macro avg       0.80      0.81      0.80      3129
weighted avg       0.82      0.81      0.81      3129

───────────────────────────────────────────────────────
CONFUSION MATRIX:
───────────────────────────────────────────────────────
(Baris = Label Asli, Kolom = Label Prediksi)
             Positif  Negatif
Positif         1573      382
Negatif          215      959


---
## Cell 10 — Visualisasi Hasil Evaluasi
> Grafik confusion matrix dan perbandingan metrik — siap untuk dijadikan gambar di Bab 4!

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Evaluasi Model Multinomial Naive Bayes\n'
    'Analisis Sentimen Dakwah Digital Kota Bandung',
    fontsize=13, fontweight='bold', y=1.02
)

LABEL_KELAS = ['Positif', 'Negatif']  # kelas Netral sudah dihapus

# ── GRAFIK 1: Confusion Matrix ───────────────────────────────
ax1 = axes[0]
sns.heatmap(
    cm,
    annot      = True,
    fmt        = 'd',
    cmap       = 'Blues',
    xticklabels= LABEL_KELAS,
    yticklabels= LABEL_KELAS,
    linewidths = 0.5,
    ax         = ax1,
    cbar_kws   = {'label': 'Jumlah'}
)
ax1.set_title('Confusion Matrix', fontweight='bold', pad=10)
ax1.set_xlabel('Label Prediksi')
ax1.set_ylabel('Label Asli')

# ── GRAFIK 2: Metrik per kelas ───────────────────────────────
from sklearn.metrics import precision_recall_fscore_support
prec_per, rec_per, f1_per, sup_per = precision_recall_fscore_support(
    y_test, y_pred, labels=LABEL_KELAS, zero_division=0
)

ax2 = axes[1]
x    = np.arange(len(LABEL_KELAS))
w    = 0.25
b1   = ax2.bar(x - w,   prec_per, w, label='Precision', color='#4C72B0', edgecolor='white')
b2   = ax2.bar(x,       rec_per,  w, label='Recall',    color='#55A868', edgecolor='white')
b3   = ax2.bar(x + w,   f1_per,   w, label='F1-Score',  color='#C44E52', edgecolor='white')
ax2.bar_label(b1, fmt='%.2f', padding=2, fontsize=8)
ax2.bar_label(b2, fmt='%.2f', padding=2, fontsize=8)
ax2.bar_label(b3, fmt='%.2f', padding=2, fontsize=8)
ax2.set_xticks(x)
ax2.set_xticklabels(LABEL_KELAS)
ax2.set_ylim(0, 1.2)
ax2.set_title('Precision / Recall / F1 per Kelas', fontweight='bold', pad=10)
ax2.set_ylabel('Skor')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.4)

# ── GRAFIK 3: Ringkasan metrik keseluruhan ───────────────────
ax3 = axes[2]
metrik_nama  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrik_nilai = [acc, prec, rec, f1]
warna        = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = ax3.bar(metrik_nama, metrik_nilai, color=warna, edgecolor='white', width=0.5)
ax3.bar_label(bars, fmt='%.4f', padding=5, fontsize=11, fontweight='bold')
ax3.set_ylim(0, 1.2)
ax3.set_title('Metrik Evaluasi Keseluruhan', fontweight='bold', pad=10)
ax3.set_ylabel('Skor')
ax3.axhline(y=0.85, color='red', linestyle='--', alpha=0.6, label='Target 85%')
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.4)

plt.tight_layout()
path_eval = os.path.join(OUTPUT_DIR, 'grafik_evaluasi_naive_bayes.png')
plt.savefig(path_eval, bbox_inches='tight', facecolor='white', dpi=150)
plt.show()
print(f'\n💾 Grafik disimpan: {path_eval}')

# ── GRAFIK 4: Confusion Matrix Normalized (%) ────────────────
fig2, ax4 = plt.subplots(figsize=(7, 6))
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_norm,
    annot      = True,
    fmt        = '.2%',
    cmap       = 'Greens',
    xticklabels= LABEL_KELAS,
    yticklabels= LABEL_KELAS,
    linewidths = 0.5,
    ax         = ax4,
)
ax4.set_title(
    'Confusion Matrix (Normalized / %)',
    fontweight='bold', pad=10
)
ax4.set_xlabel('Label Prediksi')
ax4.set_ylabel('Label Asli')
plt.tight_layout()
path_cm_norm = os.path.join(OUTPUT_DIR, 'grafik_confusion_matrix_normalized.png')
plt.savefig(path_cm_norm, bbox_inches='tight', facecolor='white', dpi=150)
plt.show()
print(f'💾 Grafik disimpan: {path_cm_norm}')


💾 Grafik disimpan: /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/grafik_evaluasi_naive_bayes.png
💾 Grafik disimpan: /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/grafik_confusion_matrix_normalized.png


---
## Cell 11 — Simpan Model & Hasil
> Simpan model yang sudah dilatih, hasil prediksi lengkap, dan laporan akhir
> ke Google Drive supaya bisa dipakai lagi tanpa training ulang.

In [11]:
import pickle
import json

# ── 1. Simpan model Naive Bayes ──────────────────────────────
path_model = os.path.join(OUTPUT_DIR, 'model_naive_bayes.pkl')
with open(path_model, 'wb') as f:
    pickle.dump(nb_model, f)
print(f'✅ Model disimpan   : {path_model}')

# ── 2. Simpan TF-IDF vectorizer ──────────────────────────────
path_tfidf = os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl')
with open(path_tfidf, 'wb') as f:
    pickle.dump(tfidf, f)
print(f'✅ TF-IDF disimpan  : {path_tfidf}')

# ── 3. Simpan hasil prediksi lengkap ke CSV ──────────────────
df_hasil                  = df.copy()
df_hasil['split']         = 'train'
df_hasil.loc[y_test.index, 'split'] = 'test'
df_hasil['label_prediksi'] = np.nan
df_hasil.loc[y_test.index, 'label_prediksi'] = y_pred

path_hasil = os.path.join(OUTPUT_DIR, 'hasil_prediksi_lengkap.csv')
df_hasil.to_csv(path_hasil, index=False, encoding='utf-8-sig')
print(f'✅ Hasil prediksi   : {path_hasil}')

# ── 4. Simpan ringkasan metrik ke JSON ───────────────────────
ringkasan = {
    'model'           : 'Multinomial Naive Bayes',
    'alpha'           : float(nb_model.alpha),
    'total_data'      : int(len(df)),
    'data_latih'      : int(X_train.shape[0]),
    'data_uji'        : int(X_test.shape[0]),
    'jumlah_fitur'    : int(X.shape[1]),
    'accuracy'        : round(float(acc),  4),
    'precision'       : round(float(prec), 4),
    'recall'          : round(float(rec),  4),
    'f1_score'        : round(float(f1),   4),
    'distribusi_label': df['sentiment_label'].value_counts().to_dict(),
}
path_json = os.path.join(OUTPUT_DIR, 'ringkasan_hasil.json')
with open(path_json, 'w', encoding='utf-8') as f:
    json.dump(ringkasan, f, ensure_ascii=False, indent=2)
print(f'✅ Ringkasan metrik : {path_json}')

# ── 5. Laporan akhir ─────────────────────────────────────────
print()
print('='*60)
print(' SCRIPT 3 SELESAI!')
print('='*60)
print(f'  Model     : Multinomial Naive Bayes (alpha=1.0)')
print(f'  Data uji  : {X_test.shape[0]:,} komentar')
print()
print(f'  ACCURACY  : {acc*100:.2f}%')
print(f'  PRECISION : {prec*100:.2f}%')
print(f'  RECALL    : {rec*100:.2f}%')
print(f'  F1-SCORE  : {f1*100:.2f}%')
print()
print('  📁 Semua file tersimpan di:')
print(f'  {OUTPUT_DIR}/')
print()
print('  ✍️  Angka-angka di atas siap ditulis di Bab 4 skripsi!')
print('='*60)

✅ Model disimpan   : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/model_naive_bayes.pkl
✅ TF-IDF disimpan  : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/tfidf_vectorizer.pkl
✅ Hasil prediksi   : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/hasil_prediksi_lengkap.csv
✅ Ringkasan metrik : /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/ringkasan_hasil.json

 SCRIPT 3 SELESAI!
  Model     : Multinomial Naive Bayes (alpha=1.0)
  Data uji  : 3,129 komentar

  ACCURACY  : 80.92%
  PRECISION : 81.80%
  RECALL    : 80.92%
  F1-SCORE  : 81.13%

  📁 Semua file tersimpan di:
  /content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2/hasil_model_binary_lexicon_tanpa_netral/

  ✍️  Angka-angka di atas siap ditulis di Bab 4 skripsi!


---
## Cell 12 — Dataset Final yang Rapi & Lengkap
> Cell ini menghasilkan **2 dataset final** yang hanya berisi kolom yang
> benar-benar diperlukan — siap dijadikan lampiran skripsi.
>
> **Output:**
> - `dataset_final_komentar.csv` → semua komentar + label + hasil prediksi
> - `dataset_final_video.csv` → semua video + topik + ringkasan + distribusi sentimen
> - `dataset_final_per_penceramah.csv` → rekap sentimen per penceramah (untuk Bab 4)

In [12]:
import pandas as pd
import numpy as np

# ============================================================
# DATASET FINAL 1: KOMENTAR
# Kolom yang benar-benar diperlukan saja
# ============================================================

# Susun kolom yang diperlukan dari df (yang sudah ada label & prediksi)
# Pastikan kolom label_prediksi dan prediksi_benar sudah ada dari Cell 8
df_final_komentar = df.copy()

# Tambahkan kolom prediksi dari hasil Cell 8
# dtype object supaya aman diisi string/bool (pandas versi baru menolak
# menyisipkan string ke kolom bertipe float)
df_final_komentar['label_prediksi'] = pd.Series(
    [pd.NA] * len(df_final_komentar), index=df_final_komentar.index, dtype='object'
)
df_final_komentar['prediksi_benar'] = pd.Series(
    [pd.NA] * len(df_final_komentar), index=df_final_komentar.index, dtype='object'
)
df_final_komentar.loc[y_test.index, 'label_prediksi'] = y_pred
df_final_komentar.loc[y_test.index, 'prediksi_benar'] = (
    y_test.values == y_pred
)
df_final_komentar['split'] = 'train'
df_final_komentar.loc[y_test.index, 'split'] = 'test'

# Pilih HANYA kolom yang diperlukan
KOLOM_KOMENTAR = [
    'channel_name',        # Nama penceramah
    'topik_utama',         # Topik dakwah video
    'comment_text',        # Komentar asli (untuk verifikasi)
    'comment_clean',       # Teks bersih (input TF-IDF)
    'comment_no_stopword', # Teks tanpa stopword (input NB)
    'sentiment_label',     # Label hasil lexicon
    'split',               # train / test
    'label_prediksi',      # Hasil prediksi model NB
    'prediksi_benar',      # Benar/salah prediksi
]

# Hanya ambil kolom yang ada
kolom_ada = [k for k in KOLOM_KOMENTAR if k in df_final_komentar.columns]
df_final_komentar = df_final_komentar[kolom_ada].reset_index(drop=True)

# Simpan
path_final_k = os.path.join(OUTPUT_DIR, 'dataset_final_komentar.csv')
df_final_komentar.to_csv(path_final_k, index=False, encoding='utf-8-sig')

print('='*60)
print('✅ DATASET FINAL 1: KOMENTAR')
print('='*60)
print(f'Jumlah baris   : {len(df_final_komentar):,}')
print(f'Jumlah kolom   : {len(df_final_komentar.columns)}')
print(f'Kolom          : {list(df_final_komentar.columns)}')
print()
print('Preview 5 baris pertama:')
display(df_final_komentar.head())
print()
print(f'Distribusi sentiment_label:')
for label, n in df_final_komentar['sentiment_label'].value_counts().items():
    pct = n / len(df_final_komentar) * 100
    print(f'  {label:<12}: {n:>6,} ({pct:.1f}%)')
print()
print(f'Distribusi split (train/test):')
print(df_final_komentar['split'].value_counts().to_string())

# ============================================================
# DATASET FINAL 2: VIDEO
# ============================================================

# Load clean_data_video dari Google Drive
path_video = os.path.join(
    '/content/drive/MyDrive/skripsi_dakwah_bandung/clean_data+denganalayv2',
    'clean_data_video.csv'
)

df_video_clean = pd.read_csv(path_video, encoding='utf-8-sig')

# Hitung distribusi sentimen per video
# Gabungkan dengan data komentar berdasarkan channel_name + topik_utama
sentimen_per_video = df_final_komentar.groupby(
    ['channel_name', 'topik_utama', 'sentiment_label']
).size().unstack(fill_value=0).reset_index()

# Pastikan semua kolom sentimen ada
for kelas in ['Positif', 'Negatif']:
    if kelas not in sentimen_per_video.columns:
        sentimen_per_video[kelas] = 0

sentimen_per_video['total_komentar'] = (
    sentimen_per_video['Positif'] +
    sentimen_per_video['Negatif']
)
sentimen_per_video['pct_positif'] = (
    sentimen_per_video['Positif'] / sentimen_per_video['total_komentar'] * 100
).round(1)
sentimen_per_video['pct_negatif'] = (
    sentimen_per_video['Negatif'] / sentimen_per_video['total_komentar'] * 100
).round(1)
sentimen_per_video['sentimen_dominan'] = sentimen_per_video[
    ['Positif', 'Negatif']
].idxmax(axis=1)

# Pilih kolom video yang diperlukan
KOLOM_VIDEO = [
    'channel_name',
    'title',
    'topik_utama',
    'ringkasan_konten',
    'kesimpulan',
    'published_at',
]
kolom_video_ada = [k for k in KOLOM_VIDEO if k in df_video_clean.columns]
df_video_final = df_video_clean[kolom_video_ada].copy()

# Gabungkan dengan distribusi sentimen
df_video_final = df_video_final.merge(
    sentimen_per_video[[
        'channel_name', 'topik_utama',
        'Positif', 'Negatif',
        'total_komentar',
        'pct_positif', 'pct_negatif',
        'sentimen_dominan'
    ]],
    on=['channel_name', 'topik_utama'],
    how='left'
)

path_final_v = os.path.join(OUTPUT_DIR, 'dataset_final_video.csv')
df_video_final.to_csv(path_final_v, index=False, encoding='utf-8-sig')

print()
print('='*60)
print('✅ DATASET FINAL 2: VIDEO + DISTRIBUSI SENTIMEN')
print('='*60)
print(f'Jumlah baris   : {len(df_video_final):,}')
print(f'Jumlah kolom   : {len(df_video_final.columns)}')
print(f'Kolom          : {list(df_video_final.columns)}')
print()
print('Preview:')
display(df_video_final[[
    'channel_name','title','topik_utama',
    'total_komentar','Positif','Negatif',
    'sentimen_dominan'
]])

# ============================================================
# DATASET FINAL 3: REKAP PER PENCERAMAH
# Ini yang paling berguna untuk Bab 4!
# ============================================================

rekap = df_final_komentar.groupby('channel_name').agg(
    total_komentar   = ('comment_text', 'count'),
    jumlah_positif   = ('sentiment_label', lambda x: (x == 'Positif').sum()),
    jumlah_negatif   = ('sentiment_label', lambda x: (x == 'Negatif').sum()),
).reset_index()

rekap['pct_positif']     = (rekap['jumlah_positif'] / rekap['total_komentar'] * 100).round(1)
rekap['pct_negatif']     = (rekap['jumlah_negatif'] / rekap['total_komentar'] * 100).round(1)
rekap['sentimen_dominan'] = rekap[['jumlah_positif','jumlah_negatif']].idxmax(axis=1).str.replace('jumlah_','')
rekap['sentimen_dominan'] = rekap['sentimen_dominan'].str.capitalize()

# Urutkan berdasarkan pct_positif tertinggi
rekap = rekap.sort_values('pct_positif', ascending=False).reset_index(drop=True)
rekap.index += 1  # Mulai dari 1

path_rekap = os.path.join(OUTPUT_DIR, 'dataset_final_per_penceramah.csv')
rekap.to_csv(path_rekap, encoding='utf-8-sig')

print()
print('='*60)
print('✅ DATASET FINAL 3: REKAP SENTIMEN PER PENCERAMAH')
print('='*60)
print(f'Jumlah penceramah: {len(rekap)}')
print()
print('Tabel rekap (urut % positif tertinggi):')
display(rekap)

# ============================================================
# RINGKASAN SEMUA OUTPUT
# ============================================================
print()
print('='*60)
print('📦 SEMUA DATASET FINAL TERSIMPAN')
print('='*60)
print(f'1. dataset_final_komentar.csv    → {len(df_final_komentar):,} baris, {len(df_final_komentar.columns)} kolom')
print(f'2. dataset_final_video.csv       → {len(df_video_final):,} baris, {len(df_video_final.columns)} kolom')
print(f'3. dataset_final_per_penceramah.csv → {len(rekap):,} baris (1 per penceramah)')
print()
print('💡 Ketiga file ini siap dijadikan lampiran skripsi!')
print('   dataset_final_per_penceramah.csv → langsung masuk Bab 4')


✅ DATASET FINAL 1: KOMENTAR
Jumlah baris   : 15,642
Jumlah kolom   : 9
Kolom          : ['channel_name', 'topik_utama', 'comment_text', 'comment_clean', 'comment_no_stopword', 'sentiment_label', 'split', 'label_prediksi', 'prediksi_benar']

Preview 5 baris pertama:


,channel_name,topik_utama,comment_text,comment_clean,comment_no_stopword,sentiment_label,split,label_prediksi,prediksi_benar
0,Ustadz Adi Hidayat,Akhlak,"Andai saja hadir juga : Aa Gym, Buya Yahya, Ha...",andai saja hadir juga aa gym buya yahya habib ...,andai hadir aa gym buya yahya habib riziek ust...,Positif,train,<NA>,<NA>
1,Ustadz Adi Hidayat,Akhlak,Saya terlahir NU dan tumbuh dg segala aturan d...,saya terlahir nu dan tumbuh dengan segala atur...,terlahir nu tumbuh aturan tata nu berkat ustad...,Positif,test,Positif,True
2,Ustadz Adi Hidayat,Akhlak,saya bersaksi guru guru kami adalah orang oran...,saya bersaksi guru guru kami adalah orang oran...,bersaksi guru guru soleh panjangkan umur mulia...,Positif,train,<NA>,<NA>
3,Ustadz Adi Hidayat,Akhlak,Teringat caption pada foto para guru ini bbrp ...,teringat caption pada foto para guru ini beber...,caption foto guru tahun apapun halaqah mu saud...,Positif,train,<NA>,<NA>
4,Ustadz Adi Hidayat,Akhlak,"""Fiqih itu luas, yang sempit itu adalah hati k...",fiqih itu luas yang sempit itu adalah hati kita,fiqih luas sempit hati,Negatif,train,<NA>,<NA>



Distribusi sentiment_label:
  Positif     :  9,771 (62.5%)
  Negatif     :  5,871 (37.5%)

Distribusi split (train/test):
split
train    12513
test      3129

✅ DATASET FINAL 2: VIDEO + DISTRIBUSI SENTIMEN
Jumlah baris   : 162
Jumlah kolom   : 12
Kolom          : ['channel_name', 'title', 'topik_utama', 'ringkasan_konten', 'kesimpulan', 'published_at', 'Positif', 'Negatif', 'total_komentar', 'pct_positif', 'pct_negatif', 'sentimen_dominan']

Preview:


,channel_name,title,topik_utama,total_komentar,Positif,Negatif,sentimen_dominan
0,Ustadz Adi Hidayat,Kunjungan para Ustadz Sahabat UAH - Ustadz Adi...,Pendidikan Islam,NaN,NaN,NaN,NaN
1,Ustadz Adi Hidayat,UAH Dilantik Jadi Stafsus? Hoax Atau Nyata? - ...,"Klarifikasi, Peran Ustadz, Prinsip Dakwah",NaN,NaN,NaN,NaN
2,Ustadz Adi Hidayat,Perbaiki Shalatmu Maka Allah Akan Permudah Uru...,Fiqih,NaN,NaN,NaN,NaN
3,Ustadz Muhammad Faizar Hidayatullah,PENAMPAKAN POCONG DI RUMAH PANJI PETUALANG,Akidah,768.0,306.0,462.0,Negatif
4,Ustadz Muhammad Faizar Hidayatullah,MANTAN AHLI SANTET - RIA PUSPITA | PERNAH MAKA...,Pengalaman Mantan Ahli Santet,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
157,Ustadz Zacky Mirza,Pengajian Bulanan MEWUJUDKAN GENERASI RABBANI ...,"Generasi Rabbani, Ramadan, Suri Tauladan",11.0,8.0,3.0,Positif
158,Ustadz Zacky Mirza,SABTU 4 APRIL 2026 - pukul 12.30 WIB_ KH.Syari...,Akhlak,2.0,1.0,1.0,Positif
159,Ustadz Subhan Bawazier,Cuma Titipan | Ustadz Subhan Bawazier,Dunia Hanyalah Titipan Allah,NaN,NaN,NaN,NaN
160,Dr. Zaidul Akbar (Jurus Sehat Rasulullah),Makanan Yang Tidak Menjadi Beban - dr.Zadul Ak...,Kesehatan Islam,23.0,15.0,8.0,Positif



✅ DATASET FINAL 3: REKAP SENTIMEN PER PENCERAMAH
Jumlah penceramah: 54

Tabel rekap (urut % positif tertinggi):


,channel_name,total_komentar,jumlah_positif,jumlah_negatif,pct_positif,pct_negatif,sentimen_dominan
1,"Ustad Taufiqurrahman, SQ",11,10,1,90.9,9.1,Positif
2,Bunda Imas Karyamah,32,29,3,90.6,9.4,Positif
3,Ustadz Aam Amiruddin (Percikan Iman),41,36,5,87.8,12.2,Positif
4,Ustadz Abdul Somad (UAS),664,581,83,87.5,12.5,Positif
5,Ustadz Handy Bonny,357,310,47,86.8,13.2,Positif
6,Ustadz Ucu Najmudin,70,59,11,84.3,15.7,Positif
7,Buya Yahya (Al-Bahjah),460,371,89,80.7,19.3,Positif
8,Ustadz Subhan Bawazier,36,29,7,80.6,19.4,Positif
9,Ustadzah Qotrunnada,487,385,102,79.1,20.9,Positif
10,Ustadzah Halimah Alaydrus,475,374,101,78.7,21.3,Positif



📦 SEMUA DATASET FINAL TERSIMPAN
1. dataset_final_komentar.csv    → 15,642 baris, 9 kolom
2. dataset_final_video.csv       → 162 baris, 12 kolom
3. dataset_final_per_penceramah.csv → 54 baris (1 per penceramah)

💡 Ketiga file ini siap dijadikan lampiran skripsi!
   dataset_final_per_penceramah.csv → langsung masuk Bab 4
